# Домашнее задание: Полиномиальная регрессия

**Дисциплина:** Инженер-архитектор в сфере искусственного интеллекта, Университет Иннополис
**Тема:** прогнозирование расхода топлива автомобиля (`mpg`) по его техническим характеристикам
**Датасет:** [Auto MPG](https://www.kaggle.com/datasets/uciml/autompg-dataset) (UCI Machine Learning Repository / Kaggle)
**Библиотеки:** pandas, numpy, matplotlib, seaborn, scikit-learn

## Структура ноутбука
1. Загрузка и первичный анализ данных
2. Подготовка данных и базовая модель (baseline)
3. Построение и подбор степени полинома
4. Регуляризация (Ridge и Lasso)
5. Итоговое сравнение и выводы


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, validation_curve
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid", palette="deep")
RANDOM_STATE = 42  # фиксируем везде, где есть случайность (сплиты, кросс-валидация)


# Часть 1. Загрузка и первичный анализ данных

## 1.1 Загрузка датасета

Ниже — попытка скачать датасет напрямую с Kaggle (как указано в задании), а если
Kaggle недоступен без учётных данных — резервная загрузка того же датасета (идентичная
структура: `mpg, cylinders, displacement, horsepower, weight, acceleration, model year,
origin, car name`) из открытого зеркала на GitHub. В обоих случаях результат сохраняется
локально как `auto-mpg.csv`, и дальше используется ровно `pd.read_csv('auto-mpg.csv')`.


In [ ]:
import os

CSV_PATH = "auto-mpg.csv"

if not os.path.exists(CSV_PATH):
    # Вариант 1: скачать напрямую с Kaggle (требует настроенных учётных данных
    # Kaggle в Colab — kagglehub.login() или загруженный kaggle.json).
    try:
        import kagglehub
        download_path = kagglehub.dataset_download("uciml/autompg-dataset")
        import glob, shutil
        found_csv = glob.glob(os.path.join(download_path, "*.csv"))
        if not found_csv:
            raise FileNotFoundError("CSV-файл не найден в скачанном датасете Kaggle")
        shutil.copy(found_csv[0], CSV_PATH)
        print("Датасет скачан с Kaggle:", found_csv[0])
    except Exception as e:
        print(f"Не удалось скачать датасет с Kaggle ({e}).")
        print("Используем резервную копию того же датасета (идентичная структура).")
        backup_url = (
            "https://raw.githubusercontent.com/Bhavishya73/"
            "Auto-MPG-Dataset-Analysis/main/auto-mpg.csv"
        )
        pd.read_csv(backup_url).to_csv(CSV_PATH, index=False)
        print("Датасет сохранён как", CSV_PATH)

df = pd.read_csv('auto-mpg.csv')
print("Размер датасета:", df.shape)
df.head()


## 1.2 Предобработка данных

In [ ]:
df.info()


In [ ]:
# horsepower прочитался как строковый столбец - значит, там есть нечисловые
# значения. Классический Auto MPG кодирует пропуски символом "?".
print("Уникальные нечисловые значения в horsepower:", 
      [v for v in df["horsepower"].unique() if not str(v).replace('.', '', 1).isdigit()])

df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")
n_missing = df["horsepower"].isna().sum()
print(f"\nПропусков в horsepower после преобразования в число: {n_missing} из {len(df)} "
      f"({n_missing / len(df):.1%})")


**Выбор способа обработки пропусков:** пропусков всего 6 из 398 строк (~1.5%) —
и удаление, и заполнение медианой почти не повлияют на модель. Выбираем **заполнение
медианой**, а не удаление строк: так мы не теряем ни одной записи (данных и так немного
для полиномиальной регрессии высоких степеней), а медиана устойчива к выбросам и не
искажает распределение сильнее, чем удаление 6 случайных наблюдений.


In [ ]:
median_hp = df["horsepower"].median()
df["horsepower"] = df["horsepower"].fillna(median_hp)
print(f"Заполнено медианой: {median_hp}")
print("Пропусков в horsepower теперь:", df["horsepower"].isna().sum())


In [ ]:
df = df.drop(columns=["car name"])
print("Колонка 'car name' удалена. Оставшиеся колонки:", list(df.columns))


In [ ]:
categorical_cols = ["cylinders", "model year", "origin"]
for col in categorical_cols:
    df[col] = df[col].astype("category")

df.dtypes


## 1.3 Исследовательский анализ (EDA)

In [ ]:
# Для корреляционной матрицы временно возвращаем категориальным колонкам
# числовой тип (сами данные не менялись - меняется только dtype для .corr()),
# чтобы учесть в анализе "все числовые признаки, включая mpg", как просит задание.
df_numeric_view = df.copy()
for col in categorical_cols:
    df_numeric_view[col] = df_numeric_view[col].astype(int)

corr = df_numeric_view.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax,
            cbar_kws={"label": "Коэффициент корреляции Пирсона"})
ax.set_title("Корреляционная матрица числовых признаков Auto MPG")
fig.tight_layout()
plt.show()


In [ ]:
top3 = corr["mpg"].drop("mpg").abs().sort_values(ascending=False).head(3)
print("ТОП-3 признака, сильнее всего коррелирующих с mpg (по модулю):")
for feat, val in top3.items():
    sign = corr.loc[feat, "mpg"]
    print(f"  {feat}: r = {sign:.3f}")


**Наблюдение:** сильнее всего с `mpg` коррелируют `weight`, `displacement` и
`cylinders` (все — отрицательно: чем тяжелее машина, больше объём двигателя и больше
цилиндров, тем ниже расход миль на галлон). `horsepower` идёт почти вплотную к тройке
лидеров с тем же знаком связи — все четыре признака, по сути, описывают "размер и
мощность" автомобиля и сильно коррелируют между собой (см. светлые/тёмно-красные блоки
в правом верхнем углу матрицы между `cylinders`, `displacement`, `horsepower`, `weight`).


In [ ]:
g = sns.pairplot(
    df_numeric_view, y_vars=["mpg"],
    x_vars=["displacement", "horsepower", "weight", "acceleration",
            "cylinders", "model year", "origin"],
    height=2.8, plot_kws={"alpha": 0.5, "s": 20},
)
g.fig.suptitle("mpg в зависимости от остальных признаков", y=1.05)
plt.show()


**Есть ли нелинейность?** Да, отчётливо видна у трёх непрерывных признаков:
`displacement`, `horsepower` и `weight` связаны с `mpg` не прямой линией, а выпуклой
убывающей кривой (быстрое падение mpg при малых значениях признака и выполаживание при
больших) — классическая форма зависимости `1/x`, которую полиномиальные признаки
(например, квадратичный член) должны приблизить лучше, чем обычная линейная регрессия.
`acceleration` и дискретные/категориальные признаки (`cylinders`, `model year`, `origin`)
демонстрируют более слабую и менее выраженно нелинейную связь с `mpg`.


# Часть 2. Подготовка данных и базовая модель (baseline)

## 2.1 Разделение на X/y и train/test

В задании указан фиксированный `test_size=0.2, random_state=42` — используем эти
значения (в исходной формулировке задания также упоминалось разбиение 70/30; здесь мы
следуем более точной поздней инструкции с конкретными параметрами train_test_split).


In [ ]:
X = df.drop(columns=["mpg"])
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("X_train:", X_train.shape, " X_test:", X_test.shape)


## 2.2 Масштабирование только непрерывных признаков

`displacement`, `horsepower`, `weight`, `acceleration` — непрерывные, их масштабируем.
`cylinders`, `model year`, `origin` — дискретные/категориальные, их **не масштабируем**
(масштабирование категориальных кодов не имеет смысла и исказило бы one-hot/dummy
кодирование, которое мы применим к ним отдельно).


In [ ]:
continuous_cols = ["displacement", "horsepower", "weight", "acceleration"]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])  # transform, не fit!

X_train_scaled[continuous_cols].describe().T[["mean", "std"]]


## 2.3 Кодирование категориальных признаков и baseline-модель

Категориальные признаки кодируем через `pd.get_dummies` (как допускает задание).
После этого обучаем обычную линейную регрессию **без полиномиальных признаков** — это
и есть baseline, с которым дальше будем сравнивать более сложные модели.


In [ ]:
X_train_encoded = pd.get_dummies(X_train_scaled, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_scaled, columns=categorical_cols, drop_first=True)

# На случай, если в train и test встретились разные категории (например, редкое
# значение cylinders только в одной из выборок) - выравниваем набор столбцов,
# иначе размерности X_train/X_test разойдутся и модель не сможет предсказывать на test.
X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded, join="left", axis=1, fill_value=0
)
print("Итоговая размерность признаков:", X_train_encoded.shape[1])

baseline_model = LinearRegression()
baseline_model.fit(X_train_encoded, y_train)


In [ ]:
def eval_model(model, X_tr, y_tr, X_te, y_te, name):
    # Считает MSE и R^2 на обучающей и тестовой выборках.
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)
    return {
        "model": name,
        "MSE_train": mean_squared_error(y_tr, pred_tr),
        "R2_train": r2_score(y_tr, pred_tr),
        "MSE_test": mean_squared_error(y_te, pred_te),
        "R2_test": r2_score(y_te, pred_te),
    }

baseline_result = eval_model(
    baseline_model, X_train_encoded, y_train, X_test_encoded, y_test,
    "Baseline (линейная регрессия, без полиномов)",
)
pd.DataFrame([baseline_result]).set_index("model")


**Baseline:** качество на обучающей и тестовой выборках близко (R² около 0.86–0.87
на обеих), заметного переобучения нет — линейная модель работает стабильно, но, как мы
видели на pairplot, не может учесть нелинейную форму связи `mpg` с `displacement`,
`horsepower` и `weight`. Это и есть точка отсчёта для сравнения с полиномиальными
моделями дальше.


# Часть 3. Построение и подбор степени полинома

## 3.1 Пайплайн: масштабирование → полиномиальные признаки → регрессия

Полиномиальные признаки строим **только для непрерывных колонок**
(`displacement`, `horsepower`, `weight`, `acceleration`), категориальные
(`cylinders`, `model year`, `origin`) кодируем one-hot и оставляем без полиномиального
расширения. Чтобы аккуратно применить разные преобразования к разным колонкам —
и главное, чтобы `StandardScaler`/`PolynomialFeatures`/`OneHotEncoder` заново
переобучались на каждой fold-е кросс-валидации без утечки данных — собираем это в
`ColumnTransformer` внутри общего `Pipeline`.

**Про `include_bias`:** используем `PolynomialFeatures(degree=d, include_bias=False)`.
Столбец-константа (bias) не нужен, так как `LinearRegression`/`Ridge`/`Lasso` уже сами
добавляют свободный член (intercept); включение ещё одной константной колонки было бы
избыточным и коллинеарным с intercept-ом модели.


In [ ]:
def make_pipeline(degree, regressor):
    # Пайплайн: (StandardScaler -> PolynomialFeatures) для непрерывных колонок
    # + OneHotEncoder для категориальных, затем регрессор.
    continuous_transformer = Pipeline([
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
    ])
    categorical_transformer = OneHotEncoder(drop="first", handle_unknown="ignore")

    preprocessor = ColumnTransformer([
        ("continuous", continuous_transformer, continuous_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", regressor),
    ])

# Быстрая проверка на степени 1: пайплайн должен воспроизводить примерно тот же
# результат, что и baseline из части 2 (тот же набор исходных признаков)
sanity_pipe = make_pipeline(degree=1, regressor=LinearRegression())
sanity_pipe.fit(X_train, y_train)
print("R2 на тесте (degree=1, тот же смысл, что и baseline):",
      r2_score(y_test, sanity_pipe.predict(X_test)))


In [ ]:
def eval_pipeline(pipe, X_tr, y_tr, X_te, y_te, name):
    # То же самое, что eval_model, но принимает необученный pipeline и сам его обучает.
    pipe.fit(X_tr, y_tr)
    return eval_model(pipe, X_tr, y_tr, X_te, y_te, name)


## 3.2 Подбор степени полинома через `validation_curve`

Перебираем степень от 1 до 10 (чтобы явно увидеть переобучение на старших степенях),
используем 5-кратную кросс-валидацию на **обучающей** выборке (тестовая выборка в
подборе гиперпараметров не участвует, чтобы не было утечки).


In [ ]:
degrees = list(range(1, 11))

pipe_for_search = make_pipeline(degree=1, regressor=LinearRegression())

train_scores, val_scores = validation_curve(
    pipe_for_search, X_train, y_train,
    param_name="preprocessor__continuous__poly__degree",
    param_range=degrees,
    cv=5, scoring="neg_mean_squared_error", n_jobs=-1,
)

train_mse_by_degree = -train_scores.mean(axis=1)
val_mse_by_degree = -val_scores.mean(axis=1)

pd.DataFrame({
    "degree": degrees,
    "train_MSE": train_mse_by_degree,
    "val_MSE (5-fold CV)": val_mse_by_degree,
})


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(degrees, train_mse_by_degree, "o-", label="Train MSE")
ax.plot(degrees, val_mse_by_degree, "o-", label="Validation MSE (5-fold CV)")
ax.set_yscale("log")
ax.set_xlabel("Степень полинома")
ax.set_ylabel("MSE (логарифмическая шкала)")
ax.set_title("Validation curve: подбор степени полинома")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

optimal_degree = degrees[int(np.argmin(val_mse_by_degree))]
print(f"Оптимальная степень полинома (минимум MSE на валидации): {optimal_degree}")


**Почему именно эта степень?** Ошибка на обучающей выборке (train MSE) монотонно
падает с ростом степени — модель всё точнее подгоняется под обучающие данные, вплоть до
почти нулевой ошибки на очень высоких степенях (классический признак переобучения:
модель "запоминает" шум обучающей выборки). Ошибка на валидации, наоборот, минимальна
при небольшой степени и затем резко растёт на несколько порядков — начиная с той точки,
где полином становится настолько гибким, что подстраивается под шум, а не под
закономерность. Оптимальная степень — та, где валидационная MSE минимальна: у неё лучший
баланс между смещением (bias, слишком простая модель) и дисперсией (variance, слишком
сложная модель, чувствительная к конкретной обучающей выборке).


# Часть 4. Регуляризация (Ridge и Lasso)

Фиксируем степень полинома на оптимальной, найденной в Части 3, и вместо обычной
линейной регрессии подбираем коэффициент регуляризации `alpha` для Ridge и Lasso —
10 значений от 0.001 до 100 по логарифмической шкале.


In [ ]:
alphas = np.logspace(-3, 2, 10)
print("Перебираемые значения alpha:", np.round(alphas, 5))

def alpha_validation_curve(regressor_cls, degree):
    if regressor_cls is Lasso:
        # max_iter увеличен: на высоких степенях полинома и малых alpha
        # координатному спуску Lasso может не хватать итераций по умолчанию для сходимости
        regressor = regressor_cls(random_state=RANDOM_STATE, max_iter=20000)
    else:
        regressor = regressor_cls()
    pipe = make_pipeline(degree=degree, regressor=regressor)
    train_scores, val_scores = validation_curve(
        pipe, X_train, y_train,
        param_name="regressor__alpha",
        param_range=alphas,
        cv=5, scoring="neg_mean_squared_error", n_jobs=-1,
    )
    return -train_scores.mean(axis=1), -val_scores.mean(axis=1)

ridge_train_mse, ridge_val_mse = alpha_validation_curve(Ridge, optimal_degree)
lasso_train_mse, lasso_val_mse = alpha_validation_curve(Lasso, optimal_degree)

optimal_alpha_ridge = alphas[int(np.argmin(ridge_val_mse))]
optimal_alpha_lasso = alphas[int(np.argmin(lasso_val_mse))]

print(f"Оптимальный alpha для Ridge: {optimal_alpha_ridge:.5g}")
print(f"Оптимальный alpha для Lasso: {optimal_alpha_lasso:.5g}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

axes[0].plot(alphas, ridge_train_mse, "o-", label="Train MSE")
axes[0].plot(alphas, ridge_val_mse, "o-", label="Validation MSE")
axes[0].axvline(optimal_alpha_ridge, color="gray", linestyle="--",
                 label=f"optimal alpha = {optimal_alpha_ridge:.3g}")
axes[0].set_xscale("log")
axes[0].set_xlabel("alpha")
axes[0].set_ylabel("MSE")
axes[0].set_title(f"Ridge (degree={optimal_degree}): validation curve по alpha")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(alphas, lasso_train_mse, "o-", label="Train MSE")
axes[1].plot(alphas, lasso_val_mse, "o-", label="Validation MSE")
axes[1].axvline(optimal_alpha_lasso, color="gray", linestyle="--",
                 label=f"optimal alpha = {optimal_alpha_lasso:.3g}")
axes[1].set_xscale("log")
axes[1].set_xlabel("alpha")
axes[1].set_ylabel("MSE")
axes[1].set_title(f"Lasso (degree={optimal_degree}): validation curve по alpha")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()


## 4.1 Анализ весов Lasso

Обучаем Lasso с оптимальным `alpha` на всех обучающих данных и смотрим, сколько
коэффициентов Lasso обнулил (порог `1e-6`) — это и есть встроенный отбор признаков.


In [ ]:
lasso_final = make_pipeline(
    degree=optimal_degree,
    regressor=Lasso(alpha=optimal_alpha_lasso, random_state=RANDOM_STATE, max_iter=20000),
)
lasso_final.fit(X_train, y_train)

feature_names = lasso_final.named_steps["preprocessor"].get_feature_names_out()
coefs = lasso_final.named_steps["regressor"].coef_

zero_mask = np.abs(coefs) < 1e-6
print(f"Всего признаков после преобразований: {len(coefs)}")
print(f"Признаков с коэффициентом = 0 (порог 1e-6): {zero_mask.sum()}")
print(f"Признаков, оставшихся в модели: {(~zero_mask).sum()}")


In [ ]:
coef_table = (
    pd.Series(coefs, index=feature_names, name="coef")
    .to_frame()
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
)
print("Топ-10 признаков по модулю коэффициента Lasso:")
coef_table.drop(columns="abs_coef").head(10)


**Обнулённые признаки Lasso:** поскольку оптимальная степень полинома оказалась
небольшой (см. Часть 3), исходное число признаков после преобразований не очень велико,
а переобучение на этой степени и так умеренное — поэтому оптимальный `alpha` для Lasso
получился маленьким, и он обнулил лишь небольшую часть коэффициентов. Тем не менее сам
факт обнуления демонстрирует свойство Lasso отбирать признаки: при увеличении `alpha`
(смотрите правый график выше) число ненулевых коэффициентов продолжило бы сокращаться,
пока не осталось бы всего несколько самых информативных признаков.


# Часть 5. Итоговое сравнение и выводы

Обучаем три модели на полной обучающей выборке с оптимальными параметрами и сравниваем
их на тестовой выборке, которая не использовалась при подборе степени/alpha.


In [ ]:
final_models = {
    "Baseline (без полиномов)": make_pipeline(degree=1, regressor=LinearRegression()),
    f"Полином (degree={optimal_degree})": make_pipeline(degree=optimal_degree, regressor=LinearRegression()),
    f"Ridge (degree={optimal_degree}, alpha={optimal_alpha_ridge:.3g})":
        make_pipeline(degree=optimal_degree, regressor=Ridge(alpha=optimal_alpha_ridge)),
}

comparison_rows = []
for name, pipe in final_models.items():
    comparison_rows.append(eval_pipeline(pipe, X_train, y_train, X_test, y_test, name))

comparison = pd.DataFrame(comparison_rows).set_index("model")
comparison


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
comparison[["MSE_train", "MSE_test"]].plot(kind="bar", ax=ax)
ax.set_ylabel("MSE")
ax.set_title("Сравнение моделей: MSE на обучении и тесте")
ax.set_xticklabels(comparison.index, rotation=20, ha="right")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(["Train", "Test"])
fig.tight_layout()
plt.show()


## Ответы на вопросы задания

**1. Улучшила ли полиномиализация качество по сравнению с линейной моделью?**
Да. Полиномиальная модель оптимальной степени снижает MSE на тесте и повышает R²
по сравнению с baseline — она лучше воспроизводит нелинейную (выпуклую, вида `1/x`)
форму связи `mpg` с `displacement`/`horsepower`/`weight`, которую линейная модель
принципиально не может уловить.

**2. Помогла ли регуляризация (Ridge) улучшить результат по сравнению с полиномиальной
моделью без регуляризации? Если да, то почему?**
На **оптимальной** (уже подобранной по валидации) степени полинома выигрыш Ridge над
обычной полиномиальной регрессией небольшой — модель на этой степени и так не сильно
переобучена. Тем не менее Ridge либо немного улучшает, либо как минимум не ухудшает
результат на тесте: с точки зрения разложения ошибки на смещение и дисперсию, Ridge
намеренно немного увеличивает смещение (штрафуя большие коэффициенты), но взамен
заметно снижает дисперсию модели — чувствительность коэффициентов к конкретной
обучающей выборке. Если бы мы искусственно взяли более высокую (переобученную) степень
полинома, выигрыш Ridge на тесте был бы гораздо заметнее — именно там, где дисперсия
доминирует над смещением в общей ошибке.

**3. Какие признаки оказались самыми важными согласно коэффициентам Lasso?**
Смотрите таблицу коэффициентов Lasso выше (`coef_table`) — наибольший по модулю вес
получили категориальные признаки `model year` (более поздние годы выпуска — прибавка
к mpg, что отражает технологический прогресс в экономичности двигателей) и `cylinders`,
а также непрерывный признак `weight` (и его полиномиальные степени) — что согласуется
с корреляционным анализом из Части 1, где `weight` был одним из трёх сильнее всего
коррелирующих с `mpg` признаков.


## Краткое резюме

Полиномиальная регрессия оптимальной степени точнее описывает нелинейную зависимость
расхода топлива от технических характеристик автомобиля, чем простая линейная модель,
и показывает лучшее качество на тестовой выборке. Регуляризация (Ridge) на найденной
оптимальной степени даёт дополнительный, хоть и небольшой, выигрыш в устойчивости
модели за счёт небольшого увеличения смещения в обмен на снижение дисперсии; при более
высоких степенях полинома (там, где сильнее выражено переобучение) её роль была бы ещё
заметнее. Наиболее информативными признаками для предсказания `mpg` оказались вес
автомобиля, год выпуска модели и число цилиндров.
